**Homework Week 8**

3/4/26

In [1]:
## Set-up

# Imports
import pandas as pd
import seaborn as sns
import plotly.express as px
import numpy as np
import zipfile
import os
from google.colab import files
from datetime import datetime

# Reading in atus files in zipped folder
# uploaded = files.upload()
with zipfile.ZipFile("FinalProjectFiles.zip", 'r') as zip_ref:
    zip_ref.extractall("FinalProjectFiles")

# All atus filepaths
folder_path = "FinalProjectFiles/FinalProjectFiles"
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".dat")]

print(len(files))
print(files)

50
['FinalProjectFiles/FinalProjectFiles/atusresp_2019.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2012.dat', 'FinalProjectFiles/FinalProjectFiles/atuscps_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2012.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2014.dat', 'FinalProjectFiles/FinalProjectFiles/atuscps_2015.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2017.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2018.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2018.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2016.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2014.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2015.dat', 'FinalProjectFiles/FinalProjectFiles/atusrost_2015.dat', 'FinalProjectFiles/FinalProjectFiles/atusresp_2011.dat', 'FinalProjectFiles/FinalProjectFiles/atusact_2012.dat', 'FinalProjectFiles/FinalProjectFiles/atuswho_2018.dat', 'FinalProjectFiles/FinalProjectFiles

In [2]:
## Reading in data

# Empty lists to store datasets
atusact_files = []
atusresp_files = []
atusrost_files = []
atuswho_files = []
atuscps_files = []

# Separating files into the four dataset categories
for f in files:
    filename = os.path.basename(f)
    if filename.startswith("atusact_"):
        atusact_files.append(f)
    elif filename.startswith("atusresp_"):
        atusresp_files.append(f)
    elif filename.startswith("atusrost_"):
        atusrost_files.append(f)
    elif filename.startswith("atuswho_"):
        atuswho_files.append(f)
    elif filename.startswith("atuscps_"):
        atuscps_files.append(f)

print(len(atusact_files), len(atusresp_files), len(atusrost_files), len(atuswho_files), len(atuscps_files))

# Function to stack datasets
def stack_atus_files(file_list):
    df_list = []
    for file_path in sorted(file_list):
        filename = os.path.basename(file_path)
        year = filename[-8:-4]
        df = pd.read_csv(file_path)
        df["year"] = int(year)
        df_list.append(df)
    stacked_df = pd.concat(df_list, ignore_index = True, sort = False) # keeps all columns
    return stacked_df

# Calling function to stack datasets
atusact_all = stack_atus_files(atusact_files)
atusresp_all = stack_atus_files(atusresp_files)
atusrost_all = stack_atus_files(atusrost_files)
atuswho_all = stack_atus_files(atuswho_files)
atuscps_all = stack_atus_files(atuscps_files)

# Checking stacked datasets
print(atusact_all.shape)
print(atusresp_all.shape)
print(atusrost_all.shape)
print(atuswho_all.shape)
print(atuscps_all.shape)

10 10 10 10 10
(2149915, 32)
(111808, 178)
(302014, 9)
(2762744, 6)
(672816, 394)


In [3]:
## Keeping relevant columns in datasets

# Activity dataset
atusact_all.columns
atusact_all = atusact_all[["TUCASEID", "TUTIER1CODE", "TRTIER2", "TUSTARTTIM", "TUSTOPTIME", "year"]]
atusact_all.head(10)

# Respondent dataset
atusresp_all.columns
atusresp_all = atusresp_all[["TUCASEID", "TUDIARYDAY", "TUDIARYDATE", "TELFS", "TRERNWA", "year"]]
atusresp_all.head(10)

# Roster dataset
atusrost_all.columns
atusrost_all = atusrost_all[["TUCASEID", "TERRP", "TEAGE", "TESEX", "year"]]
atusrost_all.head(10)

# Who dataset
atuswho_all.columns
atuswho_all = atuswho_all[["TUCASEID", "TUWHO_CODE", "year"]]
atuswho_all.head(10)

# CPS dataset
atuscps_all.columns
atuscps_all = atuscps_all[["TUCASEID", "TULINENO", "GESTFIPS", "GTMETSTA", "HEFAMINC", "HUFAMINC", "PTDTRACE", "HRMONTH", "year"]]
atuscps_all.head(10)

,TUCASEID,TULINENO,GESTFIPS,GTMETSTA,HEFAMINC,HUFAMINC,PTDTRACE,HRMONTH,year
0,20100101100019,1,42,1,-1,8.0,2,11,2010
1,20100101100019,2,42,1,-1,8.0,2,11,2010
2,20100101100019,3,42,1,-1,8.0,2,11,2010
3,20100101100019,4,42,1,-1,8.0,2,11,2010
4,20100101100019,5,42,1,-1,8.0,2,11,2010
5,20100101100020,1,12,1,-1,-3.0,1,11,2010
6,20100101100020,2,12,1,-1,-3.0,6,11,2010
7,20100101100045,1,17,1,-1,-3.0,1,11,2010
8,20100101100045,2,17,1,-1,-3.0,1,11,2010
9,20100101100045,3,17,1,-1,-3.0,1,11,2010


In [4]:
## Activity dataset cleaning

# Converting start and stop time variables to numeric
atusact_all["TUSTARTTIM"] = pd.to_datetime(atusact_all["TUSTARTTIM"], format = '%H:%M:%S', errors = 'coerce')
atusact_all["TUSTOPTIME"] = pd.to_datetime(atusact_all["TUSTOPTIME"],  format = '%H:%M:%S', errors = 'coerce')

# Converting start and stop times to minutes since midnight
atusact_all["start_minute"] = atusact_all["TUSTARTTIM"].dt.hour*60 + atusact_all["TUSTARTTIM"].dt.minute
atusact_all["stop_minute"]  = atusact_all["TUSTOPTIME"].dt.hour*60 + atusact_all["TUSTOPTIME"].dt.minute

# Adding column for total activity duration
atusact_all["duration"] = atusact_all["stop_minute"] - atusact_all["start_minute"]
atusact_all.loc[atusact_all["duration"] < 0, "duration"] += 24*60

# Adding child care and household tasks indicator columns
atusact_all["child_care"] = atusact_all["TRTIER2"].isin([301, 302, 303]).astype(int)
atusact_all["household_task"] = (atusact_all["TUTIER1CODE"] == 2).astype(int)

# Dropping unnecessary columns
# atusact_all = atusact_all.drop(columns = ["start_minute", "stop_minute", "TUSTARTTIM", "TUSTOPTIME", "TUTIER1CODE", "TRTIER2"])

# Collapsing to a single row per main respondent
atusact_final = (
    atusact_all
    .groupby(["TUCASEID", "year"], as_index = False)
    .agg(
        child_care_duration = ("duration", lambda x: np.sqrt(x[atusact_all.loc[x.index, "child_care"] == 1].sum())),
        household_task_duration = ("duration", lambda x: np.sqrt(x[atusact_all.loc[x.index, "household_task"] == 1].sum()))
    )
)

print(atusact_final.shape)
print(atusact_final.head(20))

(111808, 4)
          TUCASEID  year  child_care_duration  household_task_duration
0   20100101100019  2010             0.000000                 0.000000
1   20100101100020  2010             0.000000                14.317821
2   20100101100045  2010             6.782330                 7.745967
3   20100101100050  2010             0.000000                15.811388
4   20100101100053  2010             0.000000                 4.472136
5   20100101100087  2010             0.000000                 3.872983
6   20100101100095  2010             0.000000                 0.000000
7   20100101100098  2010             0.000000                 5.000000
8   20100101100117  2010             0.000000                 1.000000
9   20100101100119  2010             0.000000                 0.000000
10  20100101100174  2010             0.000000                 5.477226
11  20100101100175  2010             5.477226                 7.745967
12  20100101100501  2010             0.000000                10.2

In [5]:
## Respondent dataset cleaning

# Exploring missing income values
((atusresp_all["TRERNWA"] == -1)).sum() # 51,842 respondents have missing income
((atusresp_all["TRERNWA"] == -1) & ((atusresp_all["TELFS"] == 1) | (atusresp_all["TELFS"] == 2))).sum() # 7,605 employed respondents have missing income
(((atusresp_all["TELFS"] == 1) | (atusresp_all["TELFS"] == 2))).sum() # 67,571 respondents are employed
# 11.25% of employed people are missing income

# Converting missing income values to NA
atusresp_all.loc[atusresp_all["TRERNWA"] == -1, "TRERNWA"] = np.nan

# Imputing 0 income for unemployed people
atusresp_all.loc[(~atusresp_all["TELFS"].isin([1, 2])) & atusresp_all["TRERNWA"].isna(), "TRERNWA"] = 0

# Adding employed indicator
atusresp_all.loc[:, "employed"] = atusresp_all["TELFS"].isin([1, 2]).astype(int)

# Converting diary date variable to date type
atusresp_all["TUDIARYDATE"] = pd.to_datetime(atusresp_all["TUDIARYDATE"].astype(str), format = "%Y%m%d")

# Calculating annual income
atusresp_all.loc[:, "annual_income"] = atusresp_all["TRERNWA"] * 26

# Dropping unnecessary columns
atusresp_final = atusresp_all.drop(columns = ["TRERNWA", "TELFS"])

print(atusresp_final.shape)
print(atusresp_final.head(20))

(111808, 6)
          TUCASEID  TUDIARYDAY TUDIARYDATE  year  employed  annual_income
0   20100101100019           1  2010-01-24  2010         1      2142400.0
1   20100101100020           1  2010-01-31  2010         1      1515592.0
2   20100101100045           3  2010-01-26  2010         1      1560000.0
3   20100101100050           5  2010-01-28  2010         0            0.0
4   20100101100053           1  2010-01-24  2010         0            0.0
5   20100101100087           6  2010-01-29  2010         0            0.0
6   20100101100095           1  2010-01-24  2010         0            0.0
7   20100101100098           4  2010-01-27  2010         0            0.0
8   20100101100117           6  2010-01-29  2010         0            0.0
9   20100101100119           7  2010-01-30  2010         0            0.0
10  20100101100174           1  2010-01-24  2010         0            0.0
11  20100101100175           1  2010-01-31  2010         0            0.0
12  20100101100501        

In [6]:
## Roster dataset cleaning

# Filtering to only include self, spouse, and children under age 18
atusrost_filtered = atusrost_all[(atusrost_all["TERRP"].isin([18, 19, 20, 22])) & ~((atusrost_all["TERRP"] == 22) & (atusrost_all["TEAGE"] >= 18))]

# Function to only include households of heterosexual married couples with at least one child
def valid_tucaseid(group):
    has_self = group["TERRP"].isin([18, 19]).any() # self
    one_self = group["TERRP"].isin([18, 19]).sum() == 1 # one self per tucaseid
    has_spouse = (group["TERRP"] == 20).any() # spouse
    one_spouse = (group["TERRP"] == 20).sum() == 1 # one spouse per tucaseid
    has_child  = (group["TERRP"] == 22).any() # at least one household child
    if not (has_self and one_self and has_spouse and one_spouse and has_child): return False # one self, one spouse, 1+ household child
    sex_self = group.loc[group["TERRP"].isin([18, 19]), "TESEX"].values[0] # sex of self
    sex_spouse = group.loc[group["TERRP"] == 20, "TESEX"].values[0] # sex of spouse
    return sex_self != sex_spouse # confirming heterosexual couple

# Identifying households of heterosexual married couples with at least one child
valid_ids = (atusrost_filtered.groupby("TUCASEID").filter(valid_tucaseid)["TUCASEID"].unique())

# Filtering households without heterosexual married couples with at least one child
atusrost_filtered = atusrost_filtered[atusrost_filtered["TUCASEID"].isin(valid_ids)]

# Collapsing to a single row per main respondent
atusrost_final = (atusrost_filtered.groupby("TUCASEID", as_index = False).agg(
        TEAGE = ("TEAGE", lambda x: x[atusrost_filtered.loc[x.index, "TERRP"].isin([18, 19])].values[0]),
        TESEX = ("TESEX", lambda x: x[atusrost_filtered.loc[x.index, "TERRP"].isin([18, 19])].values[0]),
        num_children = ("TERRP", lambda x: (x == 22).sum()),
        year = ("year", "first")
    )
)

print(atusrost_final.shape)
print(atusrost_final.head(20))

(28240, 5)
          TUCASEID  TEAGE  TESEX  num_children  year
0   20100101100045     20      1             1  2010
1   20100101100557     33      1             1  2010
2   20100101100603     48      2             2  2010
3   20100101100712     41      2             3  2010
4   20100101100837     48      2             2  2010
5   20100101100890     37      1             1  2010
6   20100101100920     42      2             1  2010
7   20100101100937     40      2             2  2010
8   20100101100986     40      1             2  2010
9   20100101101001     41      2             2  2010
10  20100101101028     45      1             2  2010
11  20100101101083     40      2             1  2010
12  20100101101097     48      1             1  2010
13  20100101101101     50      1             2  2010
14  20100101101122     31      2             2  2010
15  20100101101140     33      2             3  2010
16  20100101101154     41      1             1  2010
17  20100101101194     37      1   

In [9]:
## CPS dataset cleaning

# Filtering to only include main respondents
atuscps_filtered = atuscps_all[atuscps_all["TULINENO"] == 1].copy()

# Converting missing income values to NA and merging two income columns
atuscps_filtered.loc[atuscps_filtered["HEFAMINC"] == -1, "HEFAMINC"] = np.nan
atuscps_filtered.loc[atuscps_filtered["HUFAMINC"] < 0, "HUFAMINC"] = np.nan
atuscps_filtered.loc[:, "HEFAMINC"] = atuscps_filtered["HEFAMINC"].fillna(atuscps_filtered["HUFAMINC"])

# Function to clean race variable
def recode_race(row):
    code = row["PTDTRACE"]
    old_coding = (row["year"] < 2012) or (row["year"] == 2012 and row["HRMONTH"] <= 4) # pre-may 2012: year < 2012, or year == 2012 and HRMONTH <= 4
    if old_coding:
        if code == 1: return "White"
        elif code == 2: return "Black"
        elif code == 4: return "Asian"
        elif code in [3, 5]: return "Other"
        else: return "Multiracial"  # codes 6-26
    else: # post-May 2012 coding
        if code == 1: return "White"
        elif code == 2: return "Black"
        elif code == 4: return "Asian"
        elif code in [3, 5]: return "Other"
        else: return "Multiracial"  # codes 6-26

# Cleaning race variable
atuscps_filtered.loc[:, "race"] = atuscps_filtered.apply(recode_race, axis = 1)

# Function to create clean income variable
def recode_income(code):
    if pd.isna(code): return np.nan
    elif code <= 6: return 1 # < $20,000
    elif code <= 11: return 2 # $20,000 - $49,999
    elif code <= 14: return 3 # $50,000 - $99,999
    elif code == 15: return 4 # $100,000 - $149,999
    else: return 5 # $150,000+

# Creating income variable
atuscps_filtered.loc[:, "income_level"] = atuscps_filtered["HEFAMINC"].apply(recode_income)
atuscps_filtered["income_level"].value_counts().sort_index()

# Dropping unnecessary columns
atuscps_final = atuscps_filtered.drop(columns = ["HUFAMINC", "TULINENO", "PTDTRACE", "HRMONTH"])

print(atuscps_final.shape)
print(atuscps_final.head(20))

(246118, 7)
          TUCASEID  GESTFIPS  GTMETSTA  HEFAMINC  year         race  \
0   20100101100019        42         1       8.0  2010        Black   
5   20100101100020        12         1       NaN  2010        White   
7   20100101100045        17         1       NaN  2010        White   
10  20100101100050        15         2       8.0  2010  Multiracial   
11  20100101100053        18         1       5.0  2010        White   
12  20100101100087        54         1      12.0  2010        White   
14  20100101100095         1         2       3.0  2010        Black   
15  20100101100098         5         2       1.0  2010        Black   
16  20100101100117        17         1       NaN  2010        Black   
18  20100101100119        32         1       2.0  2010        Black   
20  20100101100174        13         2       NaN  2010        Black   
22  20100101100175        51         1       7.0  2010        White   
27  20100101100501        24         1      16.0  2010        Bla

In [11]:
## Merging atus datasets and exporting

# Merging
atus_all = (
    atusrost_final
    .merge(atusresp_final, on = ["TUCASEID", "year"], how = "inner")
    .merge(atusact_final, on = ["TUCASEID", "year"], how = "inner")
    .merge(atuscps_final, on = ["TUCASEID", "year"], how = "inner")
)

# Exporting
atus_all.to_csv("atus_all.csv", index = False)

print(atus_all.shape)
print(atus_all.head(20))

AttributeError: 'list' object has no attribute 'download'